# MACC Data

---

### package imports and basic functions

---

In [1]:
import os
import gc
import sys
import glob
import shutil
import json
import random
import datetime
import importlib
import itertools
import numpy as np
from scipy import spatial
import scipy.sparse as sparse
import scipy.stats as stats
import pandas as pd
import nibabel as nib
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
from tqdm.auto import tqdm
from urllib.parse import urlparse
import requests
import zipfile
from pathlib import Path
import polars as pl
# import globus_sdk


In [2]:
from spectranorm import snm

In [3]:
class MyNumpyEncoder(json.JSONEncoder):
    def default(self, obj):
        if isinstance(obj, np.integer):
            return int(obj)
        elif isinstance(obj, np.floating):
            return float(obj)
        elif isinstance(obj, np.ndarray):
            return obj.tolist()
        else:
            return super(MyEncoder, self).default(obj)


def ensure_dir(file_name):
    os.makedirs(os.path.dirname(file_name), exist_ok=True)
    return file_name


def list_dirs(path=os.getcwd()):
    files = glob.glob(os.path.join(path, '*'))
    files = [x for x in files if os.path.isdir(x)]
    return files


def file_exists(file_name, path_name=os.getcwd()):
    return os.path.isfile(os.path.join(path_name, file_name))


def write_json(json_obj, file_path):
    with open(file_path, 'w') as outfile:
        json.dump(json_obj, outfile, sort_keys=True, indent=4,
                  cls=MyNumpyEncoder)
    return json_obj


def load_json(file_path):
    with open(file_path, 'r') as infile:
        return json.load(infile)


def write_np(np_obj, file_path):
    with open(file_path, 'wb') as outfile:
        np.save(outfile, np_obj)


## Extracting data

---

In [58]:
# data_info_df = pd.read_csv("/home/sina/storage/Normative_Modeling/data/csv/MACC_demogr_with_recon_all.csv")  # data provided by Tian Fang
# data_info_df = pd.read_csv("/mnt/nas/CSC22/Yeolab/Data/AIBL/users_data/Sina/MACC.csv")  # data provided by Zhang Chen
data_info_df = pd.read_csv("/mnt/nas/CSC22/Yeolab/Data/AIBL/users_data/Sina/MACC_update_dx.csv")  # data provided by Zhang Chen
data_info_df.shape


(3125, 6)

In [ ]:
data_info_df.head(10)

In [35]:
data_info_df[["scanner"]].value_counts(dropna=False)

scanner              
Siemens Tim Trio_MACC    1338
Siemens Prisma_MACC        69
Name: count, dtype: int64

In [36]:
data_info_df[["diagnosis"]].value_counts(dropna=False)

diagnosis                                
Alzheimer's dementia                         432
Cognitive impairment no dementia             388
No cognitive impairment                      338
Vascular cognitive impairment no dementia    143
Vascular dementia                            106
Name: count, dtype: int64

In [41]:
data_info_df[['Scanner_info']].value_counts(dropna=False)

Scanner_info       
NaN                    1617
SIEMENS/TrioTim/3.0    1420
SIEMENS/Prisma/3.0       88
Name: count, dtype: int64

In [43]:
data_info_df[pd.notna(data_info_df["Scan_path"])][["Scanner_info"]].value_counts(dropna=False)

Scanner_info       
SIEMENS/TrioTim/3.0    1410
SIEMENS/Prisma/3.0       87
NaN                       5
Name: count, dtype: int64

In [44]:
data_info_df[pd.notna(new_data_info_df["Scan_path"])][['DX',]].value_counts(dropna=False)


DX   
AD       465
CIND     404
CN       357
VCIND    148
VAD      114
NaN       14
Name: count, dtype: int64

In [46]:
data_info_df[pd.notna(new_data_info_df["Scan_path"])][['DX',"Scanner_info"]].value_counts(dropna=False)

DX     Scanner_info       
AD     SIEMENS/TrioTim/3.0    426
CIND   SIEMENS/TrioTim/3.0    375
CN     SIEMENS/TrioTim/3.0    346
VCIND  SIEMENS/TrioTim/3.0    144
VAD    SIEMENS/TrioTim/3.0    106
AD     SIEMENS/Prisma/3.0      37
CIND   SIEMENS/Prisma/3.0      27
NaN    SIEMENS/TrioTim/3.0     13
CN     SIEMENS/Prisma/3.0      11
VAD    SIEMENS/Prisma/3.0       8
VCIND  SIEMENS/Prisma/3.0       4
AD     NaN                      2
CIND   NaN                      2
NaN    NaN                      1
Name: count, dtype: int64

In [ ]:
list(data_info_df["Scan_path"][pd.notna(new_data_info_df["Scan_path"])][:10])

In [ ]:
macc_valid_subjects_dict = {}

for _, row in tqdm(data_info_df.iterrows()):
    if pd.notna(row["Scan_path"]):
        key = "_".join([str(row["RID"]), str(row["Scan_path"].split("/")[-2])])
        macc_valid_subjects_dict[key] = {
            "unique_id": key,
            "participant_id": str(row["RID"]),
            "session_id": str(row["Scan_path"].split("/")[-2]),
            "sex": row["Sex"],
            "age": row["Age"],
            "scan_path": row["Scan_path"],
            "diagnosis": row["DX"],
        }

len(macc_valid_subjects_dict), list(macc_valid_subjects_dict.items())[:1]


In [55]:
# Store high-resolution thickness for each individual in a separate file
for idx, subject in enumerate(tqdm(macc_valid_subjects_dict)):
    sub_dir = f"{idx:02d}"[-2:]
    macc_valid_subjects_dict[subject]["subject_index"] = idx
    freesurfer_directory = macc_valid_subjects_dict[subject]["scan_path"]
    
    thickness_fslr_output = f"/home/sina/storage/Normative_Modeling/data/fs_LR_32k/MACC/{sub_dir}/{subject}.thickness.fslr.npy"

    if not Path(thickness_fslr_output).exists():
        # Compute fslr thickness
        transformed_fslr_thickness = snm.utils.nitools.compute_fslr_thickness(freesurfer_directory)
        np.save(
            ensure_dir(thickness_fslr_output),
            transformed_fslr_thickness.astype(np.float32)
        )


  0%|          | 0/1502 [00:00<?, ?it/s]

In [ ]:
%%time
for key in tqdm(macc_valid_subjects_dict):
    idx = macc_valid_subjects_dict[key]["subject_index"]
    subject = macc_valid_subjects_dict[key]["unique_id"]
    macc_valid_subjects_dict[key]["thickness"] = np.load(
        f"/home/sina/storage/Normative_Modeling/data/fs_LR_32k/MACC/{(idx%100):02d}/{subject}.thickness.fslr.npy",
    ).mean()

len(macc_valid_subjects_dict), list(macc_valid_subjects_dict.items())[:1]


In [57]:
eno_items = [
    "lh.orig.nofix", "rh.orig.nofix",
]

# Compute Euler Number
for idx, subject in enumerate(tqdm(macc_valid_subjects_dict)):
    if (subject in macc_valid_subjects_dict) and ("euler_no" not in macc_valid_subjects_dict[subject]):
        sub_dir = f"{idx:02d}"[-2:]
        freesurfer_directory = macc_valid_subjects_dict[subject]["scan_path"]

        # Compute euler number
        macc_valid_subjects_dict[subject]["euler_no"] = snm.utils.nitools.compute_total_euler_number(
            Path(freesurfer_directory)
        )


  0%|          | 0/1502 [00:00<?, ?it/s]

In [ ]:
# add diagnosis information
for _, row in tqdm(data_info_df.iterrows()):
    if pd.notna(row["Scan_path"]):
        key = "_".join([str(row["RID"]), str(row["Scan_path"].split("/")[-2])])
        macc_valid_subjects_dict[key]["diagnosis"] = row["DX"]

len(macc_valid_subjects_dict), list(macc_valid_subjects_dict.items())[:1]


In [ ]:
len(macc_valid_subjects_dict), list(macc_valid_subjects_dict.items())[:1]

In [65]:
import joblib

dataset = "MACC"

joblib.dump(macc_valid_subjects_dict, ensure_dir(f"/home/sina/storage/Normative_Modeling/data/datasets/{dataset}/subjects.joblib"))


['/home/sina/storage/Normative_Modeling/data/datasets/MACC/subjects.joblib']

In [ ]:
import joblib

# Load the dictionary
valid_subjects_dict = joblib.load(
    f"/home/sina/storage/Normative_Modeling/data/datasets/{dataset}/subjects.joblib"
)

len(valid_subjects_dict), list(valid_subjects_dict.items())[:1]


In [67]:
# No need to make dataframe for MACC here, the dictionary will be enough for now.